# Optimizing Strategy Parameters

In [ ]:
???

<table style="width:100%; height:90%">
      <tr>
    <th>Parametrize the Strategy</th>
    <th>Optimizing Limits' Parameters</th>
  </tr>
  <tr>
    <td><img src="src/07_Code_Regression Strategy Limits X.png" alt="Parametrize the Strategy" style="width:100%"></td>
    <td><img src="src/07_Table_Optimize BG Default Defaults.png" alt="Optimizing Limits' Parameters" style="width:100%"></td>
  </tr>
</table>

## Load the model

In [1]:
import pickle

with open('models/model_dt_regression.pkl', 'rb') as f:
    model_dt = pickle.load(f)
    
model_dt

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


## Load the data

In [2]:
import pandas as pd

df = pd.read_excel('data/Microsoft_LinkedIn_Processed.xlsx', index_col=0, parse_dates=['Date'])
df

,Open,High,Low,Close,Volume,change_tomorrow,change_tomorrow_direction
Date,,,,,,,
2016-12-08,56.325228,56.582507,55.902560,56.058762,21220800,1.549143,UP
2016-12-09,56.214968,56.959234,56.169027,56.940857,27349400,0.321692,UP
2016-12-12,56.803028,57.244073,56.711145,57.124622,20198100,1.286112,UP
2016-12-13,57.427836,58.273172,57.188938,57.868881,35718900,-0.478622,DOWN
2016-12-14,57.887258,58.300739,57.455399,57.593227,30352700,-0.159789,DOWN
...,...,...,...,...,...,...,...
2023-03-09,255.820007,259.559998,251.580002,252.320007,26653400,-1.500467,DOWN
2023-03-10,251.080002,252.789993,247.600006,248.589996,28321800,2.099087,UP
2023-03-13,247.399994,257.910004,245.729996,253.919998,33339700,2.634307,UP


# Simple Investment Strategy

### Create Strategy class

In [4]:
from backtesting import Strategy, Backtest

/home/mlovera/dev/algorithmic-trading/.venv/lib/python3.12/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [5]:
class Regression(Strategy):
    def init(self):
        self.model = model_dt
        self.already_bought = False

    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :]
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        if forecast_tomorrow > 1 and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow < -5 and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass

### Create Backtest class

In [6]:
df_explanatory = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

In [7]:
bt = Backtest(df_explanatory, Regression,
              cash=10000, commission=.002, exclusive_orders=True)

### Run backtesting with specific values

In [8]:
results = bt.run()

/tmp/ipykernel_14827/695169841.py:1: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  results = bt.run()


### Interpret backtesting results

In [9]:
results.to_frame(name='Values').loc[:'Return [%]']

,Values
Start,2016-12-08 00:00:00
End,2023-03-15 00:00:00
Duration,2288 days 00:00:00
Exposure Time [%],93.908629
Equity Final [$],68877.877714
Equity Peak [$],70444.832457
Commissions [$],3892.129998
Return [%],588.778777


## Parametrize the Investment Strategy

### Create Strategy class

In [ ]:
from backtesting import Strategy, Backtest

In [16]:
class Regression(Strategy):
    limit_buy = 1
    limit_sell = -5
    
    def init(self):
        self.model = model_dt
        self.already_bought = False

    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :]
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        if forecast_tomorrow > self.limit_buy and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow < self.limit_sell and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass

### Create Backtest class

In [17]:
bt = Backtest(df_explanatory, Regression,
              cash=10000, commission=.002, exclusive_orders=True)

## Optimize backtesting with multiple combinations

In [18]:
list_limits_buy = list(range(0, 11, 1))
list_limits_buy

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [19]:
list_limits_sell = list(range(0, -11, -1))
list_limits_sell

[0, -1, -2, -3, -4, -5, -6, -7, -8, -9, -10]

In [20]:
%%time

results = bt.optimize(
    limit_buy = list_limits_buy, limit_sell = list_limits_sell,
    maximize='Return [%]', return_heatmap=True
)

/home/mlovera/dev/algorithmic-trading/.venv/lib/python3.12/site-packages/backtesting/_stats.py:156: RuntimeWarning: divide by zero encountered in log
  equity_log_returns = np.log(equity[1:] / equity[:-1])
/home/mlovera/dev/algorithmic-trading/.venv/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/mlovera/dev/algorithmic-trading/.venv/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/mlovera/dev/algorithmic-trading/.venv/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and i

CPU times: user 2.66 s, sys: 145 ms, total: 2.8 s
Wall time: 1min 8s


/home/mlovera/dev/algorithmic-trading/.venv/lib/python3.12/site-packages/backtesting/backtesting.py:1545: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = self.run(**dict(zip(heatmap.index.names, best_params)))


In [21]:
results[0]

Start                     2016-12-08 00:00:00
End                       2023-03-15 00:00:00
Duration                   2288 days 00:00:00
Exposure Time [%]                    99.61929
Equity Final [$]               13842316.29135
Equity Peak [$]                13842316.29135
Commissions [$]                 3565406.93412
Return [%]                       138323.16291
Buy & Hold Return [%]               373.50315
Return (Ann.) [%]                   217.88827
Volatility (Ann.) [%]                81.70124
CAGR [%]                            121.80549
Sharpe Ratio                          2.66689
Sortino Ratio                        15.20096
Calmar Ratio                         11.13541
Alpha [%]                         138238.1233
Beta                                  0.22768
Max. Drawdown [%]                   -19.56714
Avg. Drawdown [%]                    -1.63685
Max. Drawdown Duration       39 days 00:00:00
Avg. Drawdown Duration        8 days 00:00:00
# Trades                          

In [24]:
results[0].to_frame(name='Values').style

,Values
Start,2016-12-08 00:00:00
End,2023-03-15 00:00:00
Duration,2288 days 00:00:00
Exposure Time [%],99.619289
Equity Final [$],13842316.291354
Equity Peak [$],13842316.291354
Commissions [$],3565406.934117
Return [%],138323.162914
Buy & Hold Return [%],373.503151
Return (Ann.) [%],217.888267


### [ ] Interpret optimization results

In [26]:
dff = results[0].reset_index()
dff

,index,0
0,Start,2016-12-08 00:00:00
1,End,2023-03-15 00:00:00
2,Duration,2288 days 00:00:00
3,Exposure Time [%],99.619289
4,Equity Final [$],13842316.291354
5,Equity Peak [$],13842316.291354
6,Commissions [$],3565406.934117
7,Return [%],138323.162914
8,Buy & Hold Return [%],373.503151
9,Return (Ann.) [%],217.888267


In [27]:
dff = dff.pivot(index='limit_buy', columns='limit_sell', values='Return [%]')

KeyError: 'limit_buy'

### DataFrame heatmaps for better reporting

In [28]:
dff.sort_index(axis=1, ascending=False)\
  .style.format(precision=0)\
    .background_gradient(vmin=dff.values.min(), vmax=dff.values.max())

TypeError: '<' not supported between instances of 'str' and 'int'